In [ ]:
import bisect

class Solution(object):
    def pathExistenceQueries(self, n, nums, maxDiff, queries):
        # 1. Sort the elements while keeping track of their original indices
        sorted_nodes = sorted((val, i) for i, val in enumerate(nums))
        
        # Map original index to its position in the sorted array
        pos_map = {orig_idx: sorted_idx for sorted_idx, (_, orig_idx) in enumerate(sorted_nodes)}
        
        # 2. Find the greedy next step for each node in the sorted array
        # nxt[i] will store the index of the furthest node to the right reachable from i
        nxt = [0] * n
        for i in range(n):
            val = sorted_nodes[i][0]
            # Find the largest index j such that sorted_nodes[j][0] <= val + maxDiff
            # bisect_right gives the first element > val + maxDiff, so subtract 1
            idx = bisect.bisect_right(sorted_nodes, (val + maxDiff, float('inf'))) - 1
            nxt[i] = idx

        # 3. Build the Sparse Table for Binary Lifting
        LOG = 18  # Since n <= 10^5, 2^17 = 131072 > n
        up = [[0] * LOG for _ in range(n)]
        
        for i in range(n):
            up[i][0] = nxt[i]
            
        for j in range(1, LOG):
            for i in range(n):
                up[i][j] = up[up[i][j-1]][j-1]
                
        ans = []
        for u, v in queries:
            pos_u = pos_map[u]
            pos_v = pos_map[v]
            
            # Since the graph is undirected, distance(u, v) == distance(v, u)
            if pos_u > pos_v:
                pos_u, pos_v = pos_v, pos_u
                
            if pos_u == pos_v:
                ans.append(0)
                continue
            
            # Check reachability: if the maximum possible jumps cannot reach pos_v, it's impossible
            max_reachable = pos_u
            for j in range(LOG - 1, -1, -1):
                max_reachable = up[max_reachable][j]
            if max_reachable < pos_v:
                ans.append(-1)
                continue
                
            # Perform binary lifting to count the minimum steps
            steps = 0
            curr = pos_u
            for j in range(LOG - 1, -1, -1):
                # If jumping 2^j steps doesn't reach or overshoot pos_v, take the jump
                if up[curr][j] < pos_v:
                    curr = up[curr][j]
                    steps += (1 << j)
            
            # From the final `curr`, one more greedy step is guaranteed to reach or cross `pos_v`
            ans.append(steps + 1)
            
        return ans